## WIP - Create attention + transformer network similar to GPT-3 

1. ~~Tokenization using BPE~~
2. ~~Embedding Layer: Positional + Token Embedding~~
3. Self-Attention Layer
4. Transformer Block
5. Training Loop
6. Model Evaluation
7. Load Pre-Trained weights
8. Fine-Tuning

In [1]:
from typing import Tuple
import tiktoken
from tiktoken import Encoding
import torch
from torch.utils.data import DataLoader, Dataset
import requests
from bs4 import BeautifulSoup

c:\Users\nikhi\OneDrive\Documentos\Repos\MLNotebooks\.venv\Lib\site-packages\torch\_subclasses\functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


## Example text Data to work with

Many texts available on public domain and documented in wikisource.  I use "A Scandal in Bohemia" by Arthur Conan Doyle.

In [2]:
url = "https://en.wikisource.org/wiki/The_Strand_Magazine/Volume_2/Issue_7/A_Scandal_in_Bohemia"
output_file = "datasets/a_scandal_in_bohemia.txt"


def fetch_story(url: str) -> str:
    response = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    # Wikisource story text lives inside div.prp-pages-output
    story_div = soup.find("div", class_="prp-pages-output")
    if not story_div:
        raise ValueError(
            "Could not find story content div (prp-pages-output) on the page.")

    # Remove invisible page-number markers
    for span in story_div.find_all("span", class_="pagenum"):
        span.decompose()

    paragraphs = story_div.find_all("p")
    text = "\n\n".join(
        p.get_text(separator=" ", strip=True) for p in paragraphs
        if p.get_text(strip=True))
    return text

text = fetch_story(url)

with open(output_file, "w", encoding="utf-8") as f:
    f.write(text)

print(f"Saved {len(text)} characters to '{output_file}'.")

Saved 46784 characters to 'datasets/a_scandal_in_bohemia.txt'.


## Tokenized Dataset

- Create a dataset class in pytorch to take a tokenizer and tokenize the data

In [3]:
class TokenizedDataset(Dataset):

    def __init__(self, text: str, tokenizer: Encoding, max_length: int,
                 stride: int) -> None:
        self.input_ids = []
        self.target_ids = []

        # encode text
        token_ids = tokenizer.encode(text)

        # sliding window of length `max_length` and stride `stride`
        # store inputs and target pairs
        for i in range(0, len(token_ids) - max_length, stride):
            input_i = token_ids[i:i + max_length]
            target_i = token_ids[i + 1:i + max_length + 1]

            self.input_ids.append(torch.tensor(input_i))
            self.target_ids.append(torch.tensor(target_i))

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, index: int) -> Tuple[int, int]:
        return self.input_ids[index], self.target_ids[index]

## DataLoader to batch and load data


In [4]:
# initialize tokenizer, create dataset, create dataloader return dataloader
def tokenized_dataloader(text: str, batch_size: int, max_length: int,
                         stride: int, shuffle: bool, drop_last: bool,
                         num_workers: int):
    tokenizer = tiktoken.get_encoding("gpt2")
    tokenized_dataset = TokenizedDataset(text, tokenizer, max_length, stride)

    tokenized_dataset_loader = DataLoader(tokenized_dataset,
                                          batch_size=batch_size,
                                          shuffle=shuffle,
                                          drop_last=drop_last,
                                          num_workers=num_workers)

    return tokenized_dataset_loader

In [6]:
with open("./datasets/a_scandal_in_bohemia.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

## Embedding Layer

### Token Embedding + Positional Embedding

- Token Embedding : Using Byte Pair Encoding (BPE) with tiktoken Library
- Positional Embedding: Absolute Positional Enbedding to distinguish location of similar tokens in context

In [ ]:

vocab_size = 50250
embedding_size = 256
batch_size = 8
max_length = 4
stride = 4

token_embedding_layer = torch.nn.Embedding(vocab_size, embedding_size)
dataloader = tokenized_dataloader(raw_text,
                                  batch_size=batch_size,
                                  max_length=max_length,
                                  stride=stride,
                                  shuffle=False,
                                  drop_last=True,
                                  num_workers=0)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print(inputs.shape)
print(targets.shape)
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

# positional embedding
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, embedding_size)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))

input_embeddings = token_embeddings + pos_embeddings
print(input_embeddings.shape)